# Import what you need

In [ ]:
threads = 2

import os
os.environ['OMP_NUM_THREADS']=str(threads)
import tensorflow as tf

# PyTorch favours OMP_NUM_THREADS in environment
import torch

# Tensorflow needs explicit cofig calls
tf.config.threading.set_inter_op_parallelism_threads(threads)
tf.config.threading.set_intra_op_parallelism_threads(threads)

In [ ]:
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import glob

* Batch_size = write the same batch_size you used as **hps['batch_size']** 

In [ ]:
X_test = tf.data.Dataset.load('01.datasets/intcoords/test')

# get batched version of dataset to feed to AAE model for prediction
X_test_batched = X_test.batch(64,drop_remainder=True)

# get numpy version for testing purposes
X_test_np = np.stack(list(X_test))

test_geom = np.moveaxis(np.stack(list(tf.data.Dataset.load('01.datasets/geoms/test'))),2,0)
test_geom.shape

mol_model = torch.jit.load('./03.model/features.pt')
torch_encoder = torch.jit.load('./03.model/encoder-norm.pt')
torch_decoder = torch.jit.load('./03.model/decoder-norm.pt')

In [ ]:
conf = './../dataset_Thermal-unfolding/trpcage_npt400_nH.pdb'
tr_full = md.load('./../dataset_Thermal-unfolding/trpcage_ds_nH.xtc',top=conf)

m = torch.jit.load('./03.model/model.pt')
lows = m(torch.tensor(tr_full.xyz)).numpy()

lows.shape

# Disentanglement 

In [ ]:
!pip install seaborn

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Disentanglement metrics optimized for a 2D latent space:
- nHSIC (normalized HSIC) with RBF kernel and median heuristic
- Mutual Information (KSG-1) with k-NN and Chebyshev norm
- Reproducible subsampling and optional standardization
- Optimized for 2D analysis with dedicated visualizations

Outputs:
  1) <base>_metrics.csv           (key metrics)
  2) <base>_space.png/pdf         (scatter/hexbin plot z1 vs z2)
  3) <base>_marginals.png/pdf     (histograms of z1 and z2)
  4) <base>_dependence.png/pdf    (visualization of nHSIC and MI)

References:
- Gretton et al., "A Kernel Statistical Test of Independence", NIPS 2008
- Kraskov et al., "Estimating mutual information", PRE 2004
"""

from __future__ import annotations

import os
import csv
from typing import Dict, Tuple, Union

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

def set_global_seed(seed: int = 42) -> None:
    """Set global RNG seeds for full(ish) reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # torch.use_deterministic_algorithms(True)  # uncomment if you need strict determinism

# ---------------------------------------------------------
# Efficient subsampling
# ---------------------------------------------------------

def subsample_rows(Z: Union[torch.Tensor, np.ndarray], max_n: int = 5000, seed: int = 42):
    """
    Subsample rows while preserving representativeness.
    For 2D, we keep it simple (random); stratification can be added if needed.
    Returns (Z_subsampled, n_original).
    """
    if isinstance(Z, torch.Tensor):
        n = Z.shape[0]
        if n <= max_n:
            return Z, n
        g = torch.Generator(device=Z.device).manual_seed(seed)
        idx = torch.randperm(n, generator=g, device=Z.device)[:max_n]
        return Z[idx], n

    elif isinstance(Z, np.ndarray):
        n = Z.shape[0]
        if n <= max_n:
            return Z, n
        rng = np.random.default_rng(seed)
        idx = rng.permutation(n)[:max_n]
        return Z[idx], n

    else:
        raise TypeError("Z must be a torch.Tensor or numpy.ndarray")

# ---------------------------------------------------------
# Robust standardization
# ---------------------------------------------------------

def zscore_columns(Z: Union[torch.Tensor, np.ndarray], robust: bool = False):
    """
    Z-score standardization (mean=0, std=1).
    If robust=True, use median and MAD instead of mean and std.
    """
    eps = 1e-12
    if isinstance(Z, torch.Tensor):
        if robust:
            median = Z.median(dim=0, keepdim=True).values
            mad = (Z - median).abs().median(dim=0, keepdim=True).values
            scale = 1.4826 * mad + eps  # MAD to std conversion
            return (Z - median) / scale
        else:
            mean = Z.mean(dim=0, keepdim=True)
            std = Z.std(dim=0, unbiased=False, keepdim=True) + eps
            return (Z - mean) / std

    elif isinstance(Z, np.ndarray):
        if robust:
            median = np.median(Z, axis=0, keepdims=True)
            mad = np.median(np.abs(Z - median), axis=0, keepdims=True)
            scale = 1.4826 * mad + eps
            return (Z - median) / scale
        else:
            mean = Z.mean(axis=0, keepdims=True)
            std = Z.std(axis=0, keepdims=True) + eps
            return (Z - mean) / std

    else:
        raise TypeError("Z must be a torch.Tensor or numpy.ndarray")

# ---------------------------------------------------------
# RBF kernel with optimized median heuristic
# ---------------------------------------------------------

def _pairwise_sq_dists(x: torch.Tensor) -> torch.Tensor:
    # Computes the full squared Euclidean distance matrix
    x2 = (x ** 2).sum(1, keepdim=True)
    dist = x2 + x2.T - 2.0 * (x @ x.T)
    return torch.clamp(dist, min=0.0)


def rbf_kernel(x: torch.Tensor, sigma: torch.Tensor | None = None, subsample_for_bandwidth: bool = True):
    """
    RBF kernel with bandwidth selection via the median heuristic.
    For large datasets, sample to estimate the bandwidth.
    Returns (K, sigma).
    """
    with torch.no_grad():
        n = x.shape[0]
        if sigma is None:
            if subsample_for_bandwidth and n > 2000:
                # Estimate bandwidth using a subset to avoid O(n^2) distance storage just for the median
                g = torch.Generator(device=x.device).manual_seed(0)
                idx = torch.randperm(n, generator=g, device=x.device)[:2000]
                xs = x[idx]
                dists_sample = _pairwise_sq_dists(xs)
                pos = dists_sample > 0
                if pos.any():
                    med = torch.median(dists_sample[pos])
                    sigma = torch.sqrt(med / 2.0 + 1e-12)
                else:
                    sigma = torch.tensor(1.0, device=x.device)
            else:
                dists_full = _pairwise_sq_dists(x)
                pos = dists_full > 0
                if pos.any():
                    med = torch.median(dists_full[pos])
                    sigma = torch.sqrt(med / 2.0 + 1e-12)
                else:
                    sigma = torch.tensor(1.0, device=x.device)

        # Build full kernel once sigma is known
        dists = _pairwise_sq_dists(x)
        K = torch.exp(-dists / (2 * sigma ** 2 + 1e-12))

    return K, sigma

# ---------------------------------------------------------
# nHSIC (fixed centering)
# ---------------------------------------------------------

def nhsic_2d(z1: torch.Tensor, z2: torch.Tensor, return_sigma: bool = False):
    """
    Compute normalized HSIC between two 1D variables.
    Uses centered Gram matrices Kc = H K H and Lc = H L H (recommended).

    Returns:
        nhsic: float in [0, 1]
        (sigma1, sigma2): bandwidths used (if return_sigma=True)
    """
    assert z1.ndim == 2 and z2.ndim == 2
    assert z1.shape[1] == 1 and z2.shape[1] == 1

    n = z1.shape[0]
    H = torch.eye(n, device=z1.device) - torch.ones(n, n, device=z1.device) / n

    with torch.no_grad():
        K, sigma1 = rbf_kernel(z1)
        L, sigma2 = rbf_kernel(z2)

        Kc = H @ K @ H
        Lc = H @ L @ H

        num = torch.trace(Kc @ Lc)
        den = torch.sqrt(torch.trace(Kc @ Kc) * torch.trace(Lc @ Lc) + 1e-12)

        nhsic = (num / den).clamp(min=0.0, max=1.0).item()

    if return_sigma:
        return nhsic, (float(sigma1), float(sigma2))
    return nhsic

# ---------------------------------------------------------
# MI (KSG-1) optimized for 2D
# ---------------------------------------------------------

def mi_ksg_2d(z1: torch.Tensor, z2: torch.Tensor, k: int = 5) -> Tuple[float, float]:
    """
    Mutual Information via the KSG-1 estimator with Chebyshev (L-infinity) norm.

    Returns:
        mi_nats: float (MI in nats)
        mi_bits: float (MI in bits)
    """
    assert z1.ndim == 2 and z2.ndim == 2
    assert z1.shape[1] == 1 and z2.shape[1] == 1

    x = z1.squeeze(1)
    y = z2.squeeze(1)
    n = x.shape[0]
    if k <= 0 or k >= n:
        raise ValueError(f"k must be in [1, n-1]; got k={k}, n={n}")

    eps = 1e-10

    with torch.no_grad():
        # Pairwise distances for Chebyshev norm
        dx = (x[:, None] - x[None, :]).abs()
        dy = (y[:, None] - y[None, :]).abs()
        dist_inf = torch.maximum(dx, dy)

        # Ignore self-distances
        inf_mask = torch.eye(n, device=x.device, dtype=torch.bool)
        dist_inf = dist_inf.masked_fill(inf_mask, float('inf'))

        # k-th nearest neighbor radius in joint space
        kth = torch.topk(dist_inf, k, largest=False).values[:, -1]
        eps_i = kth + eps  # small margin to avoid tie issues

        # Neighbor counts in marginals (exclude the point itself)
        nx = (dx < eps_i[:, None]).sum(dim=1) - 1
        ny = (dy < eps_i[:, None]).sum(dim=1) - 1

        psi = torch.digamma
        I = psi(torch.tensor(float(k), device=x.device)) \
            + psi(torch.tensor(float(n), device=x.device)) \
            - (psi(nx.float() + 1.0) + psi(ny.float() + 1.0)).mean()

        mi_nats = float(torch.clamp(I, min=0.0))
        mi_bits = mi_nats / np.log(2.0)

    return mi_nats, mi_bits

# ---------------------------------------------------------
# Descriptive statistics
# ---------------------------------------------------------

def compute_2d_statistics(Z_t: torch.Tensor) -> Dict[str, float]:
    """Compute descriptive statistics for a 2D latent space."""
    with torch.no_grad():
        z1 = Z_t[:, 0]
        z2 = Z_t[:, 1]
        z1_mean = float(z1.mean())
        z2_mean = float(z2.mean())
        z1_std = float(z1.std())
        z2_std = float(z2.std())
        # Safe Pearson correlation (avoid NaN when std=0)
        if z1_std == 0.0 or z2_std == 0.0:
            pearson = 0.0
        else:
            pearson = float(torch.corrcoef(Z_t.T)[0, 1])
        stats = {
            'z1_mean': z1_mean,
            'z1_std': z1_std,
            'z1_min': float(z1.min()),
            'z1_max': float(z1.max()),
            'z2_mean': z2_mean,
            'z2_std': z2_std,
            'z2_min': float(z2.min()),
            'z2_max': float(z2.max()),
            'pearson_corr': pearson,
        }
    return stats

# ---------------------------------------------------------
# 2D visualizations
# ---------------------------------------------------------

def plot_latent_space_2d(Z_t: torch.Tensor, out_base: str, title: str = "Latent Space 2D") -> None:
    """Hexbin scatter plot of the latent space with density coloring."""
    Z_np = Z_t.detach().cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 8))

    hb = ax.hexbin(Z_np[:, 0], Z_np[:, 1], gridsize=50, cmap='viridis',
                   mincnt=1, alpha=0.8, edgecolors='none')

    ax.set_xlabel('z₁', fontsize=14)
    ax.set_ylabel('z₂', fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Density', fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()


def plot_marginal_distributions(Z_t: torch.Tensor, out_base: str) -> None:
    """Histograms of marginal distributions."""
    Z_np = Z_t.detach().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for i, ax in enumerate(axes):
        ax.hist(Z_np[:, i], bins=50, alpha=0.7, color='steelblue',
                edgecolor='black', density=True)
        ax.set_xlabel('z₁' if i == 0 else 'z₂', fontsize=14)
        ax.set_ylabel('Density', fontsize=14)
        ax.set_title('Marginal Distribution z₁' if i == 0 else 'Marginal Distribution z₂',
                     fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()


def plot_dependence_metrics(nhsic: float, mi_nats: float, mi_bits: float, out_base: str) -> None:
    """Bar plot for dependence metrics."""
    fig, ax = plt.subplots(figsize=(10, 6))

    metrics = ['nHSIC', 'MI (nats)', 'MI (bits)']
    values = [float(nhsic), float(mi_nats), float(mi_bits)]

    bars = ax.barh(metrics, values, alpha=0.85, edgecolor='black', linewidth=1.2)

    # Add labels on bars
    for bar, val in zip(bars, values):
        width = bar.get_width()
        ax.text(width + max(values) * 0.02 + 1e-9, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', ha='left', va='center', fontsize=12, fontweight='bold')

    ax.set_xlabel('Value', fontsize=14)
    ax.set_title('Dependence Metrics (z₁, z₂)', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim(0, max(max(values) * 1.2, 0.1))

    plt.tight_layout()
    plt.savefig(f"{out_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{out_base}.pdf", bbox_inches='tight')
    plt.close()

# ---------------------------------------------------------
# CSV export
# ---------------------------------------------------------

def export_2d_metrics_csv(
    path: str,
    n_original: int,
    n_used: int,
    nhsic: float,
    mi_nats: float,
    mi_bits: float,
    stats: Dict[str, float],
    sigma_info: Tuple[float, float] | None = None,
) -> None:
    """Export all metrics to a CSV file."""
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])

        # Dataset info
        w.writerow(["n_original", n_original])
        w.writerow(["n_used", n_used])
        w.writerow(["n_latents", 2])

        # Dependence metrics
        w.writerow(["nhsic_z1_z2", f"{nhsic:.6f}"])
        w.writerow(["mi_z1_z2_nats", f"{mi_nats:.6f}"])
        w.writerow(["mi_z1_z2_bits", f"{mi_bits:.6f}"])

        # Descriptive statistics
        for key, val in stats.items():
            w.writerow([key, f"{val:.6f}"])

        # Bandwidth info (if available)
        if sigma_info is not None:
            w.writerow(["rbf_sigma_z1", f"{sigma_info[0]:.6f}"])
            w.writerow(["rbf_sigma_z2", f"{sigma_info[1]:.6f}"])

# ---------------------------------------------------------
# Main 2D analysis pipeline
# ---------------------------------------------------------

def analyze_2d_latent_space(
    Z: Union[np.ndarray, torch.Tensor],
    out_dir: str = ".",
    device: str | None = None,
    standardize: bool = True,
    robust_standardize: bool = False,
    max_n: int = 5000,
    seed: int = 42,
    k_mi: int = 5,
    base_name: str = "latent_2d",
) -> Dict[str, object]:
    """
    Full pipeline for analyzing a 2D latent space.

    Args:
        Z: array/tensor [n, 2] - 2D latent space
        out_dir: output directory
        device: 'cpu', 'cuda', or None (auto)
        standardize: apply z-score
        robust_standardize: use median/MAD instead of mean/std
        max_n: max samples for subsampling
        seed: global seed for reproducibility
        k_mi: k-nearest neighbors for MI estimator
        base_name: prefix for output files

    Returns:
        dict with metrics and file paths
    """
    os.makedirs(out_dir, exist_ok=True)
    set_global_seed(seed)

    # Validate input
    if isinstance(Z, (np.ndarray, torch.Tensor)):
        if Z.shape[1] != 2:
            raise ValueError(f"Z must have 2 columns, found {Z.shape[1]}")
    else:
        raise TypeError("Z must be a numpy.ndarray or torch.Tensor")

    print("[INFO] 2D latent space analysis")
    print(f"[INFO] Original shape: {Z.shape}")

    # 1) Standardization (optional)
    if standardize:
        Z = zscore_columns(Z, robust=robust_standardize)
        print(f"[INFO] Standardization: {'robust (MAD)' if robust_standardize else 'classical (std)'}")

    # 2) Subsampling
    Z_sub, n_original = subsample_rows(Z, max_n=max_n, seed=seed)
    print(f"[INFO] Subsampling: {n_original} → {Z_sub.shape[0]} samples")

    # 3) Convert to torch
    if isinstance(Z_sub, np.ndarray):
        Z_t = torch.from_numpy(Z_sub)
    else:
        Z_t = Z_sub
    Z_t = Z_t.float()

    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    Z_t = Z_t.to(device)
    print(f"[INFO] Device: {device}")

    n_used = Z_t.shape[0]

    # 4) Metrics
    print("[INFO] Computing nHSIC…")
    nhsic, sigma_info = nhsic_2d(Z_t[:, 0:1], Z_t[:, 1:2], return_sigma=True)

    print(f"[INFO] Computing MI (k={k_mi})…")
    mi_nats, mi_bits = mi_ksg_2d(Z_t[:, 0:1], Z_t[:, 1:2], k=k_mi)

    print("[INFO] Computing descriptive statistics…")
    stats = compute_2d_statistics(Z_t)

    # 5) CSV export
    csv_path = os.path.join(out_dir, f"{base_name}_metrics.csv")
    export_2d_metrics_csv(csv_path, n_original, n_used, nhsic, mi_nats, mi_bits, stats, sigma_info)
    print(f"[EXPORT] Metrics: {csv_path}")

    # 6) Visualizations
    print("[INFO] Generating plots…")

    latent_base = os.path.join(out_dir, f"{base_name}_space")
    plot_latent_space_2d(Z_t, latent_base)
    print(f"[EXPORT] Latent space: {latent_base}.png/pdf")

    marginal_base = os.path.join(out_dir, f"{base_name}_marginals")
    plot_marginal_distributions(Z_t, marginal_base)
    print(f"[EXPORT] Marginals: {marginal_base}.png/pdf")

    metrics_base = os.path.join(out_dir, f"{base_name}_dependence")
    plot_dependence_metrics(nhsic, mi_nats, mi_bits, metrics_base)
    print(f"[EXPORT] Dependence metrics: {metrics_base}.png/pdf")

    # 7) Summary
    print("\n" + "=" * 60)
    print("2D ANALYSIS RESULTS")
    print("=" * 60)
    print(f"nHSIC(z₁, z₂):     {nhsic:.6f}  [0=indep, 1=dependent]")
    print(f"MI(z₁, z₂) nats:   {mi_nats:.6f}")
    print(f"MI(z₁, z₂) bits:   {mi_bits:.6f}")
    print(f"Pearson corr:      {stats['pearson_corr']:.6f}")
    print("=" * 60 + "\n")

    return {
        "nhsic": nhsic,
        "mi_nats": mi_nats,
        "mi_bits": mi_bits,
        "sigma_z1": sigma_info[0],
        "sigma_z2": sigma_info[1],
        "statistics": stats,
        "csv_path": csv_path,
        "n_original": n_original,
        "n_used": n_used,
    }


In [ ]:
lows2 = np.loadtxt("../03.ASMSA_manuscript_sidechains_mixtureofgasussians/lows2.txt")

In [ ]:
# 2) Lancia la pipeline
res = analyze_2d_latent_space(
    lows2,
    out_dir="outputs",         # cartella di output
    device='cpu',               # 'cuda' o 'cpu' (None = auto)
    standardize=True,          # z-score
    robust_standardize=False,  # usa mediana/MAD se True
    max_n=10000,                # sottocampiona a 5k
    seed=42,                   # riproducibilità
    k_mi=4,                    # K per MI (KSG-1)
    base_name="latent_2d"      # prefisso file
)

print("RISULTATI:", res)

In [ ]:
import numpy as np
h = np.loadtxt("./../01.ASMSA_manuscript_sidechains_normal/04.mtd_files/HILLS")
l = np.loadtxt("./../01.ASMSA_manuscript_sidechains_normal/lows.txt")

In [ ]:
cv1, cv2 = h[:, 1], h[:, 2]
l1, l2 = l[:, 0], l[:, 1]

# Properties
* Color the latent space above with the variables calculated in this section to explore the computed properties in the low dimentinal space

In [ ]:
import matplotlib.pyplot as plt

xmin = min(l1.min(), cv1.min())
xmax = max(l1.max(), cv1.max())
ymin = min(l2.min(), cv2.min())
ymax = max(l2.max(), cv2.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im1 = axes[0].hist2d(l1, l2, bins=50, range=[[xmin, xmax], [ymin, ymax]], cmap='Blues')
axes[0].set_title("Density l")
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel("l1")
axes[0].set_ylabel("l2")

# ---- Densità h ----
im2 = axes[1].hist2d(cv1, cv2, bins=50, range=[[xmin, xmax], [ymin, ymax]], cmap='Oranges')
axes[1].set_title("Density h")
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_xlabel("cv1")
axes[1].set_ylabel("cv2")

# ---- Sovrapposizione ----
# Creiamo due mappe 2D con gli stessi bin
H_l, xedges, yedges = np.histogram2d(l1, l2, bins=50, range=[[xmin, xmax], [ymin, ymax]])
H_h, _, _ = np.histogram2d(cv1, cv2, bins=[xedges, yedges])

# Plot sovrapposto con trasparenze
axes[2].imshow(H_l.T, origin='lower', extent=[xmin, xmax, ymin, ymax], cmap='Blues', alpha=0.6)
axes[2].imshow(H_h.T, origin='lower', extent=[xmin, xmax, ymin, ymax], cmap='Oranges', alpha=0.6)
axes[2].set_title("Densities")
axes[2].set_aspect('equal', adjustable='box')
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")

plt.tight_layout()
plt.show()


## RMSD & RG

In [ ]:
rg = md.compute_rg(tr_full)
base = md.load(conf)
rmsd = md.rmsd(tr_full,base[0])
cmap = plt.get_cmap('rainbow')
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.scatter(lows[:,0],lows[:,1],marker='.',c=rmsd,cmap=cmap,s=1)
plt.colorbar(cmap=cmap)
plt.title("RMSD")
plt.subplot(122)
plt.scatter(lows[:,0],lows[:,1],marker='.',c=rg,cmap=cmap,s=1)
plt.colorbar(cmap=cmap)
plt.title("RMSD")
plt.savefig("rmsd-rg-norm.png")

## Torsions

In [ ]:
def wrap180_deg(a_rad):
    """Convert radians to degrees and wrap into [-180, 180)."""
    return (np.degrees(a_rad) + 180.0) % 360.0 - 180.0

# === INPUT ===
resid_pdb = 6   # residue number in the PDB
t = tr_full     # preloaded MDTraj trajectory
t.superpose(t[0], atom_indices=t.top.select("name CA"))

# === Find residue by PDB number ===
res = [r for r in t.top.residues if r.resSeq == resid_pdb][0]

# === Compute torsions ===
phi_idx, phi_rad = md.compute_phi(t)
psi_idx, psi_rad = md.compute_psi(t)
chi1_idx, chi1_rad = md.compute_chi1(t)
chi2_idx, chi2_rad = md.compute_chi2(t)

def get_col_for_res(torsion_idx, residue):
    """Return the torsion column corresponding to the given residue."""
    for i, atoms in enumerate(torsion_idx):
        resids = [t.top.atom(a).residue.index for a in atoms]
        if residue.index in resids:
            return i
    return None

i_phi  = get_col_for_res(phi_idx,  res)
i_psi  = get_col_for_res(psi_idx,  res)
i_chi1 = get_col_for_res(chi1_idx, res)
i_chi2 = get_col_for_res(chi2_idx, res)

phi  = wrap180_deg(phi_rad[:,  i_phi])  if i_phi  is not None else None
psi  = wrap180_deg(psi_rad[:,  i_psi])  if i_psi  is not None else None
chi1 = wrap180_deg(chi1_rad[:, i_chi1]) if i_chi1 is not None else None
chi2 = wrap180_deg(chi2_rad[:, i_chi2]) if i_chi2 is not None else None

    # Translate angles greater than 150 degrees to the negative range
chi1[chi1 > 150] -= 360
chi2[chi2 > 150] -= 360

# === Create figure grid ===
fig, axs = plt.subplots(3, 4, figsize=(18, 12))
plt.subplots_adjust(wspace=0.35, hspace=0.35)

# φ vs ψ (colored by φ)
axs[0, 0].scatter(phi, psi, s=1, c=phi, cmap="rainbow")
axs[0, 0].set_title("φ vs ψ (colored by φ)")
axs[0, 0].set_xlabel("φ (°)")
axs[0, 0].set_ylabel("ψ (°)")

# φ vs ψ (colored by ψ)
axs[0, 1].scatter(phi, psi, s=1, c=psi, cmap="rainbow")
axs[0, 1].set_title("φ vs ψ (colored by ψ)")
axs[0, 1].set_xlabel("φ (°)")
axs[0, 1].set_ylabel("ψ (°)")

# χ1 vs χ2 (colored by χ1)
axs[0, 2].scatter(chi1, chi2, s=1, c=chi1, cmap="rainbow")
axs[0, 2].set_title("χ1 vs χ2 (colored by χ1)")
axs[0, 2].set_xlabel("χ1 (°)")
axs[0, 2].set_ylabel("χ2 (°)")

# χ1 vs χ2 (colored by χ2)
axs[0, 3].scatter(chi1, chi2, s=1, c=chi2, cmap="rainbow")
axs[0, 3].set_title("χ1 vs χ2 (colored by χ2)")
axs[0, 3].set_xlabel("χ1 (°)")
axs[0, 3].set_ylabel("χ2 (°)")

# Embedding colored by φ
axs[1, 0].scatter(lows[:, 0], lows[:, 1], c=phi, cmap="rainbow", s=1)
axs[1, 0].set_title("Embedding (colored by φ)")
axs[1, 0].set_xlabel("PC1")
axs[1, 0].set_ylabel("PC2")

# Embedding colored by ψ
axs[1, 1].scatter(lows[:, 0], lows[:, 1], c=psi, cmap="rainbow", s=1)
axs[1, 1].set_title("Embedding (colored by ψ)")
axs[1, 1].set_xlabel("PC1")
axs[1, 1].set_ylabel("PC2")

# Embedding colored by χ1
axs[1, 2].scatter(lows[:, 0], lows[:, 1], c=chi1, cmap="rainbow", s=1)
axs[1, 2].set_title("Embedding (colored by χ1)")
axs[1, 2].set_xlabel("PC1")
axs[1, 2].set_ylabel("PC2")

# Embedding colored by χ2
axs[1, 3].scatter(lows[:, 0], lows[:, 1], c=chi2, cmap="rainbow", s=1)
axs[1, 3].set_title("Embedding (colored by χ2)")
axs[1, 3].set_xlabel("PC1")
axs[1, 3].set_ylabel("PC2")

# Leave last row empty for future plots or notes
for ax in axs[2, :]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("torsions_norm.png")

## Alpha elics
* **Traj** must be the tranining .xtc and .pdb

In [ ]:
dssp = md.compute_dssp(tr_full, simplified=True) 
alpha_content_per_frame = np.mean(dssp == 'H', axis=1)
average_alpha_helix_content = np.mean(alpha_content_per_frame)

print(f"Avarage alpha elics content: {average_alpha_helix_content:.3f}")
plt.scatter(lows[:, 0], lows[:, 1], c=alpha_content_per_frame, cmap="rainbow", s=1)
plt.savefig("dssp_norm.png")

## Contact pairs
* **x**: residue number.
* **y**:  Ca, Cb or whatever belonging with X, the user wish to compute. 

In [ ]:
for atom in tr_full.topology.atoms:
    print(atom.index, atom.name, atom.residue)

In [ ]:
res1 = 2
name1 = "N"
res2 = 19
name2 = "N"

atom_indices = (tr_full.topology.select(f'"resid {res1} and name {name1}"')[0],  
                tr_full.topology.select(f'"resid {res2} and name {name1}"')[0])

distances = md.compute_distances(tr_full, [atom_indices]) 
plt.scatter(lows[:, 0], lows[:, 1], c=distances, cmap="rainbow", s=1)
plt.savefig(f'"distances_norm_{res1}-{name1}_{res2}-{name2}.png"')

## Angles
* **x**: same as above.
* **y**:  same as above

In [ ]:
atom_indices = tr_full.topology.select("resid 2 and name N")[0], \
               tr_full.topology.select("resid 12 and name C")[0], \
               tr_full.topology.select("resid 19 and name O")[0]

# Radiants
angles = md.compute_angles(tr_full, [atom_indices])  

# Degree
angles_deg = np.rad2deg(angles[:, 0])
plt.scatter(lows[:, 0], lows[:, 1], c=angles_deg, cmap="rainbow", s=1)

## plane
* **x**: same as above.
* **y**:  same as above

In [ ]:
atom1 = tr_full.topology.select("resid 2 and name N")[0]
atom2 = tr_full.topology.select("resid 8 and name C")[0]
atom3 = tr_full.topology.select("resid 13 and name C")[0]
atom4 = tr_full.topology.select("resid 19 and name O")[0]

# Radiants
dihedrals = md.compute_dihedrals(tr_full, [[atom1, atom2, atom3, atom4]])
# Degree
dihedrals_deg = np.rad2deg(dihedrals[:, 0])  
plt.scatter(lows[:, 0], lows[:, 1], c=dihedrals_deg, cmap="rainbow", s=1)

## Save all pictures

In [ ]:
os.makedirs("0x.picture", exist_ok=True)

for file in glob.glob("*.png"):
        shutil.move(file, os.path.join("03.pictures", file))